In [ ]:
###
# セットアップ
###

import sys
import os
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

ROOT_PATH = Path("/content/drive/MyDrive/cnn-hands-on")

if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

In [ ]:
###
# 1. 必要なライブラリのインポート
###

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

from utils.dataloader import get_cat_dog_dataloaders

# GPUが使える場合はGPUを、使えない場合はCPUを使用する設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用するデバイス: {device}")

train_loader, val_loader, test_loader = get_cat_dog_dataloaders(
    root_path=ROOT_PATH,
    data_size="large",
    batch_size=32
)

In [ ]:
###
# 2. CNN
###

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 128x128の画像を入力
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        
        # プーリング2回で 128 -> 64 -> 32 に縮小
        self.fc1 = nn.Linear(in_features=32 * 32 * 32, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=2) 

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        
        x = torch.flatten(x, 1)
        
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        
        return x

model = SimpleCNN().to(device)
print(model)

In [ ]:
###
# 3. 学習の実行
###

# 第5回で学ぶ：損失関数と最適化
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10 # 授業時間の都合上、まずは5エポック程度で回します
train_loss_list = []
val_loss_list = []
val_acc_list = []

best_val_loss = float("inf")

os.makedirs(ROOT_PATH / "models", exist_ok=True)
save_path = ROOT_PATH / "models" / "01_intro.pth"

print("学習開始")
for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    running_train_loss = 0.0
    
    # バッチごとにデータを取り出して学習
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # 第6回で学ぶ：学習の4ステップ
        optimizer.zero_grad()               # 1. 勾配のリセット
        outputs = model(images)             # 2. 予測（順伝播）
        loss = criterion(outputs, labels)   # 3. 誤差の計算
        loss.backward()                     # 4. 誤差逆伝播
        optimizer.step()                    # 5. パラメータ更新
        
        running_train_loss += loss.item()
        
    # 1エポック分の平均Lossを記録
    epoch_train_loss = running_train_loss / len(train_loader)
    train_loss_list.append(epoch_train_loss)
    
    # --- Validation ---
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_val_loss = running_val_loss / len(val_loader)
    epoch_val_acc = 100 * correct / total

    val_loss_list.append(epoch_val_loss)
    val_acc_list.append(epoch_val_acc)

    if epoch_val_loss < best_val_loss: # lossは小さい方がいい
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), save_path)
        mark = "Best Model Saved"
    else:
        mark = ""

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {epoch_train_loss:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} | "
          f"Val Acc: {epoch_val_acc:.2f}% {mark}")
 
print("学習完了")

In [ ]:
###
# 5. 結果の描画 (グラフ)
###

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# --- Loss（誤差）の推移 ---
ax1.plot(train_loss_list, label='Train Loss', color='blue', marker='o')
ax1.plot(val_loss_list, label='Validation Loss', color='orange', marker='o')


ax1.set_title('Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# --- Accuracy（正答率）の推移 ---
ax2.plot(val_acc_list, label='Validation Accuracy', color='green', marker='o')
ax2.set_title('Accuracy Curve')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()